<a href="https://colab.research.google.com/github/pjh06625-ai/kream/blob/kream/%EA%B1%B0%EB%9E%98%EC%95%A1_baseline_%EC%98%88%EC%B8%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install prophet

In [15]:
import os
os.listdir('/content')

['.config', '거래액데이터2.csv', 'sample_data']

In [16]:
import pandas as pd

# 1. 파일 불러오기
df = pd.read_csv('/content/거래액데이터2.csv',thousands=',')

df = df.rename(columns={'일자': 'ds', '총거래액': 'y'})

# 3. 날짜 데이터 형식으로 확실히 변환 (이것도 꼭 해줘야 안전합니다)
df['ds'] = pd.to_datetime(df['ds'])

# 이름이 잘 바뀌었는지 윗부분 확인
display(df.head())

,ds,y
0,2025-01-01,8907
1,2025-01-02,7592
2,2025-01-03,5147
3,2025-01-04,4539
4,2025-01-05,5706


In [19]:
import pandas as pd
import numpy as np
from prophet import Prophet

# ==========================================
# 1. 데이터 준비 및 전처리
# ==========================================

df = df.copy()

# 달력 구조 반영을 위한 파생변수 (주말이면 1, 평일이면 0)
df['is_weekend'] = df['ds'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)


# ==========================================
# 2. Prophet 모델 세팅 및 학습
# ==========================================
m = Prophet(
    seasonality_mode='multiplicative', # [핵심] 거래액 규모에 비례해 변동폭도 커지는 현상 반영
    changepoint_prior_scale=0.1,       # 트렌드 변화를 조금 더 유연하게 캐치 (기본값 0.05)
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

# 한국 공휴일 및 주말 변수 등록
m.add_country_holidays(country_name='KR')
m.add_regressor('is_weekend')

# 모델 학습
m.fit(df)


# ==========================================
# 3. 미래 예측 (다음 달 30일치 예측)
# ==========================================
future = m.make_future_dataframe(periods=30, freq='D')
future['is_weekend'] = future['ds'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

forecast = m.predict(future)


# ==========================================
# 4. 월별 집계 및 병합 (작성해주신 로직 최적화)
# ==========================================
# 4-1. 실제값 월별 집계
actual_df = df[['ds', 'y']].copy()
actual_df['month'] = actual_df['ds'].dt.to_period('M')
actual_monthly = actual_df.groupby('month')['y'].sum().round(0).astype(int).reset_index()
actual_monthly.columns = ['month', 'actual_gmv']

# 4-2. 예측값 월별 집계 (yhat 사용)
forecast_df = forecast[['ds', 'yhat']].copy()
forecast_df['month'] = forecast_df['ds'].dt.to_period('M')
forecast_monthly = forecast_df.groupby('month')['yhat'].sum().round(0).astype(int).reset_index()
forecast_monthly.columns = ['month', 'next_month_forecast']

# 4-3. 병합
result = pd.merge(actual_monthly, forecast_monthly, on='month', how='outer')
result = result.sort_values('month').reset_index(drop=True)


# ==========================================
# 5. 오차율 계산 및 최근 3개월 평균 보정
# ==========================================
# 실제값이 존재하는 데이터에 대해서만 오차율 계산
result['error_rate'] = (result['actual_gmv'] - result['next_month_forecast']) / result['next_month_forecast']

# 최근 3개월 평균 오차율 산출 (실제값이 있는 마지막 3개 행 기준)
avg_error = result['error_rate'].dropna().tail(3).mean()
avg_error_6m = result['error_rate'].dropna().tail(6).mean()
print(f'▶ 최근 3개월 평균 오차율: {avg_error:.1%}\n')
print(f'▶ 최근 6개월 평균 오차율: {avg_error_6m:.1%}\n')

# 미래 예측치(실제값이 없는 달) 보정 적용
result['adjusted_forecast'] = result['next_month_forecast'] * (1 + avg_error)
result['adjusted_forecast'] = result['adjusted_forecast'].round(0)

# 결과 확인
print(result.to_string(index=False))

▶ 최근 3개월 평균 오차율: 1.3%

▶ 최근 6개월 평균 오차율: 0.4%

  month  actual_gmv  next_month_forecast  error_rate  adjusted_forecast
2025-01    195259.0               202293   -0.034771           204988.0
2025-02    205279.0               201988    0.016293           204679.0
2025-03    225439.0               230564   -0.022228           233636.0
2025-04    209730.0               209018    0.003406           211803.0
2025-05    251141.0               246473    0.018939           249757.0
2025-06    210177.0               213728   -0.016615           216575.0
2025-07    209157.0               209054    0.000493           211839.0
2025-08    206338.0               205824    0.002497           208566.0
2025-09    228119.0               226691    0.006299           229711.0
2025-10    222964.0               229088   -0.026732           232140.0
2025-11    249220.0               244390    0.019763           247646.0
2025-12    254330.0               256658   -0.009070           260077.0
2026-01    222851.